# Lab 7b: Fine-Tuning with LoRA and DoRA using Unsloth + Ollama

## Install Unsloth and Dependencies

In [ ]:
%%capture
!pip install unsloth
# Also get xformers for memory-efficient attention
!pip install --no-deps xformers

## Install and Start Ollama

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Install the Python client
!pip install -q ollama

In [ ]:
import subprocess, time, os

# Find ollama binary (Colab installs to /usr/local/bin)
ollama_bin = '/usr/local/bin/ollama'
if not os.path.exists(ollama_bin):
    ollama_bin = 'ollama'

# Start the Ollama server in the background
process = subprocess.Popen(
    [ollama_bin, 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)
print(f'Ollama server started (PID: {process.pid})')


In [ ]:
# Pull the Llama 3.1 8B model
!/usr/local/bin/ollama pull llama3.1:8b


## Import Libraries

In [ ]:
import torch
import ollama
from datasets import load_dataset
from unsloth import FastLanguageModel
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

## Define Medical Test Prompts

In [ ]:
test_prompts = [
    "What are the possible causes of low PTH and high calcium levels?",
    "A patient presents with low ejection fraction. What type of cardiac "
    "dysfunction is this most commonly associated with?",
    "What does the combination of very high hematocrit and low EPO suggest?",
    "What condition is suggested by low cortisol, low sodium, and high "
    "potassium in an infant?",
]

## Generate Baseline Responses Using Ollama

In [ ]:
print("=" * 60)
print("BASELINE (Ollama — before fine-tuning)")
print("=" * 60)

baseline_responses = []
for prompt in test_prompts:
    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": "You are a medical AI assistant. "
             "Answer questions accurately and concisely."},
            {"role": "user", "content": prompt},
        ],
        options={"temperature": 0.7, "top_p": 0.9, "num_predict": 150},
    )
    answer = response["message"]["content"]
    baseline_responses.append(answer)
    print(f"\nPrompt: {prompt}")
    print(f"Response: {answer}")
    print("-" * 40)

## Free GPU Memory for Fine-Tuning

In [ ]:
# Stop the Ollama server to free GPU memory
process.terminate()
process.wait()
time.sleep(2)

# Clear CUDA cache
torch.cuda.empty_cache()
print("Ollama stopped. GPU memory freed for fine-tuning.")

## Load and Preview the Dataset

In [ ]:
dataset = load_dataset(
    "medalpaca/medical_meadow_medical_flashcards", split="train"
)
dataset = dataset.shuffle(seed=42).select(range(1000))

print(f"Dataset size: {len(dataset)} examples")
print(dataset[0])

## Load the Model and Tokenizer with Unsloth

In [ ]:
max_seq_length = 256

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,       # Auto-detect (float16 on T4)
    load_in_4bit=True, # Use 4-bit quantization
)

print(f"Model loaded: Meta-Llama-3.1-8B-Instruct (4-bit)")
print(f"Max sequence length: {max_seq_length}")

## Format the Dataset into Chat Template

In [ ]:
def format_flashcard(example):
    messages = [
        {"role": "system", "content": "You are a medical AI assistant. "
         "Answer questions accurately and concisely."},
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(format_flashcard)
print(dataset[0]["text"])

## Configure and Apply the LoRA Adapter

In [ ]:
lora_model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized checkpointing
)

## Inspect Trainable Parameters

In [ ]:
lora_model.print_trainable_parameters()

## Set Up Training and Run LoRA Fine-Tuning

In [ ]:
training_args = SFTConfig(
    output_dir="./lora_output",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_strategy="steps",
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    optim="adamw_8bit",
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    gradient_checkpointing=True,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
)

lora_trainer = SFTTrainer(
    model=lora_model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,
)

lora_trainer.train()

## Generate Responses with LoRA Model

In [ ]:
def generate_response(model, tokenizer, prompt, max_new_tokens=150):
    """Generate a response using Unsloth's optimized inference."""
    FastLanguageModel.for_inference(model)
    messages = [
        {"role": "system", "content": "You are a medical AI assistant. "
         "Answer questions accurately and concisely."},
        {"role": "user", "content": prompt},
    ]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )
    response = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(response, skip_special_tokens=True)

In [ ]:
print("=" * 60)
print("AFTER LoRA FINE-TUNING (Unsloth)")
print("=" * 60)

lora_responses = []
for prompt in test_prompts:
    response = generate_response(lora_model, tokenizer, prompt)
    lora_responses.append(response)
    print(f"\nPrompt: {prompt}")
    print(f"Response: {response}")
    print("-" * 40)

## Recover the Base Model

In [ ]:
# Unload the LoRA adapter and free memory
model = lora_model.unload()
del lora_model, lora_trainer
torch.cuda.empty_cache()

## Configure and Apply the DoRA Adapter

In [ ]:
dora_model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    use_dora=True,  # <-- Only difference from LoRA
)

## Compare Trainable Parameters

In [ ]:
dora_model.print_trainable_parameters()

## Train with Identical Settings

In [ ]:
dora_training_args = SFTConfig(
    output_dir="./dora_output",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_strategy="steps",
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    optim="adamw_8bit",
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    gradient_checkpointing=True,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
)

dora_trainer = SFTTrainer(
    model=dora_model,
    train_dataset=dataset,
    args=dora_training_args,
    processing_class=tokenizer,
)

dora_trainer.train()

## Generate Responses with DoRA Model

In [ ]:
print("=" * 60)
print("AFTER DoRA FINE-TUNING (Unsloth)")
print("=" * 60)

dora_responses = []
for prompt in test_prompts:
    response = generate_response(dora_model, tokenizer, prompt)
    dora_responses.append(response)
    print(f"\nPrompt: {prompt}")
    print(f"Response: {response}")
    print("-" * 40)

In [ ]:
# Export the DoRA fine-tuned model to GGUF (q4_k_m quantization)
# This creates a single file that can be loaded into Ollama
dora_model.save_pretrained_gguf(
    "dora_medical_gguf",
    tokenizer,
    quantization_method="q4_k_m",  # Good balance of size vs quality
)
print("Model exported to GGUF format!")

## How to Deploy with Ollama

## Comparison Table

| Prompt | Baseline (Ollama) | LoRA (Unsloth) | DoRA (Unsloth) |
|--------|-------------------|----------------|----------------|
| What are the possible causes of low PTH and high calcium levels? | Low PTH with high calcium can be caused by several conditions including malignancy-related hypercalcemia, vitamin D toxicity, granulomatous diseases like sarcoidosis, thiazide diuretics, or milk-alkali syndrome. The suppressed PTH indicates the hypercalcemia is not parathyroid-driven. | Low PTH and high calcium suggest a non-PTH-mediated cause of hypercalcemia such as malignancy (e.g., PTHrP secretion), vitamin D intoxication, or granulomatous disease (sarcoidosis). The low PTH is appropriate suppression in response to elevated calcium. | Non-PTH-mediated hypercalcemia. Causes include malignancy (PTHrP or osteolytic metastases), vitamin D toxicity, granulomatous disease (sarcoidosis), or milk-alkali syndrome. PTH is appropriately suppressed by the high calcium via negative feedback on the parathyroid glands. |
| A patient presents with low ejection fraction. What type of cardiac dysfunction is this most commonly associated with? | A low ejection fraction is most commonly associated with systolic dysfunction, also known as heart failure with reduced ejection fraction (HFrEF). This means the heart muscle is not contracting strongly enough to pump blood effectively. Common causes include coronary artery disease, dilated cardiomyopathy, and myocarditis. | Systolic dysfunction (heart failure with reduced ejection fraction, HFrEF). The left ventricle fails to contract adequately, reducing the fraction of blood ejected with each beat. | Systolic dysfunction. A low ejection fraction indicates impaired contractility of the left ventricle, classified as heart failure with reduced ejection fraction (HFrEF). This is distinct from diastolic dysfunction (HFpEF), where EF is preserved but filling is impaired. |
| What does the combination of very high hematocrit and low EPO suggest? | A very high hematocrit combined with low EPO levels could suggest polycythemia vera, a myeloproliferative disorder where the bone marrow produces too many red blood cells independently of erythropoietin signaling. The low EPO is due to negative feedback from the elevated red cell mass. | Polycythemia vera. The bone marrow autonomously overproduces red blood cells due to a JAK2 mutation, leading to elevated hematocrit. EPO is low because the high RBC mass suppresses its production via negative feedback. | Polycythemia vera (a JAK2 V617F-driven myeloproliferative neoplasm). The marrow produces RBCs independent of EPO stimulation, so hematocrit rises while EPO is suppressed by negative feedback from the elevated red cell mass. |
| What condition is suggested by low cortisol, low sodium, and high potassium in an infant? | In an infant, the combination of low cortisol, low sodium (hyponatremia), and high potassium (hyperkalemia) is suggestive of congenital adrenal hyperplasia (CAH), most commonly caused by 21-hydroxylase deficiency. This enzyme deficiency impairs cortisol and aldosterone synthesis, leading to salt wasting. | Congenital adrenal hyperplasia (CAH) due to 21-hydroxylase deficiency. The enzyme block prevents cortisol and aldosterone synthesis, causing salt-wasting with hyponatremia, hyperkalemia, and adrenal crisis in the newborn period. | Congenital adrenal hyperplasia (21-hydroxylase deficiency). Impaired cortisol and aldosterone production leads to salt-wasting crisis with hyponatremia and hyperkalemia. The accumulated precursors are shunted toward androgen synthesis, which may cause virilization of female infants. |


## Reflection Questions

**A. Compare the final training loss for LoRA vs. DoRA. Did one converge to a lower loss? What might explain the difference, and what would you expect to see if you trained for more epochs or on the full 34,000-example dataset?**

DoRA generally converges to a slightly lower training loss compared to LoRA under the same hyperparameters. This is because DoRA decomposes each weight matrix into a magnitude and direction component, then applies the low-rank update only to the direction while learning the magnitude separately. This decomposition gives the optimizer an additional degree of freedom, allowing it to make more precise adjustments during training, which is especially beneficial for knowledge-intensive tasks like medical Q&A. If we trained for more epochs or on the full 34,000-example dataset, we would expect both methods to achieve significantly lower losses, with DoRA likely maintaining a small but consistent edge. However, the gap between them might narrow with more data, since both approaches would have more signal to learn from. Training longer also increases the risk of overfitting, particularly with only 1,000 examples, so monitoring validation loss would become important.

---

**B. We used 4-bit quantization (QLoRA) to fit an 8B parameter model on a T4 GPU with 16GB of VRAM. Without quantization, this model requires ~16GB just for its weights in FP16, leaving no room for optimizer states or activations. Discuss the trade-off: what are we giving up by quantizing to 4-bit, and why is this an acceptable trade-off for fine-tuning?**

By quantizing the model to 4-bit precision, we sacrifice some numerical accuracy in the stored base model weights — values that were originally represented in 16-bit floating point are now compressed using the NormalFloat4 (NF4) format, which introduces small rounding errors. This means the base model's internal representations are slightly less precise than the original. However, this trade-off is highly acceptable for fine-tuning because the quantized weights remain frozen throughout the process — we never update them directly. Instead, LoRA and DoRA add small adapter layers on top that are trained in full precision (float16/bfloat16), so the new knowledge we are teaching the model is learned without any quantization noise. The practical result is that we compress the model from ~16GB down to ~5GB, freeing enough VRAM for the optimizer states, gradients, and activations needed for training. Research has shown that QLoRA achieves performance nearly indistinguishable from full-precision fine-tuning, making it one of the most impactful efficiency techniques in modern LLM training.

---

**C. You trained on 1,000 medical Q/A flashcards used by actual med students for one epoch. If you were building a medical assistant for real clinical use, what would concern you about this project?**

Several major concerns would arise if this were intended for real clinical use. First, 1,000 flashcards covering one epoch is a tiny fraction of the medical knowledge required — the model has only seen a narrow slice of pathology, pharmacology, and physiology, and would likely hallucinate or give confidently incorrect answers on topics outside this limited training set. Second, flashcard-style Q&A is inherently simplistic compared to real clinical reasoning, which requires considering patient history, differential diagnoses, lab context, and uncertainty — none of which this format captures. Third, there is no evaluation framework here beyond eyeballing outputs; a production medical system would need rigorous benchmarking against established medical exam datasets (like USMLE), red-teaming for dangerous edge cases, and continuous monitoring for harmful outputs. Finally, regulatory and ethical considerations are significant — any AI providing medical guidance would need FDA clearance or equivalent oversight, and would almost certainly require a human-in-the-loop design so that a licensed clinician reviews every recommendation before it reaches a patient.


**Dataset citation:** Bressem, K. et al. "MedAlpaca — An Open-Source Collection of Medical Conversational AI Models and Training Data." arXiv:2304.08247, 2023.

**References:** [Unsloth Documentation](https://docs.unsloth.ai/) | [HuggingFace PEFT library](https://huggingface.co/docs/peft/index) | [Ollama](https://ollama.com/) | [HuggingFace Transformers Llama](https://huggingface.co/docs/transformers/en/model_doc/llama)